In [ ]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt

# Force Open3D to use X11 compatibility layer instead of native Wayland
import os
os.environ['XDG_SESSION_TYPE'] = 'x11'

In [9]:
# Reads the point cloud file and returns an instance of the PointCLoud class
# .ply file <-- Polygon File Format, contains both point cloud and mesh data
pcd_o3d = o3d.io.read_point_cloud("data/seychelles-beach.ply")
print(pcd_o3d)

# Visualizes the point cloud
o3d.visualization.draw_geometries([pcd_o3d])

# Voxel downsampling
print("Downsample the point cloud with a voxel of 0.05")
downpcd = pcd_o3d.voxel_down_sample(voxel_size=0.05)
o3d.visualization.draw_geometries([downpcd])

PointCloud with 5000000 points.
Downsample the point cloud with a voxel of 0.05


In [ ]:
# Vertex normals estimation
print("Recompute the normal of the downsampled point cloud")
# To estimate normals for point cloud, the computer needs to look at the neighbours of every single point to figure out surface orientation
# KDTreeSearchParamHybrid is the logic that determines which neighbour to pick
# k-d tree (k-dimensional tree) used to organize points in a multip-dimentional space
# instead of checking every point in the cloud to find neighbour, it finds nearby points, and discards entire sections of space that are too far away

# radius search: looks for all points within a specific radius / distance
# knn (k-nearest neighbour): looks for a fixed number of closest points
# KDTreeSearchParamHybrid(radius=0.1, max_nn=30)): meaning, find all points within a 0.1 unit radius, but stop looking once you've found 30 points
# max_nn: maximum nearest neighbours (caps the computational cost per point)

# other options:
# KDTreeSearchParamKNN(k = 20): finds exactly k neighbours, regardless of how far away they are
# best for: uniformly dense scans
# KDTreeSearchParamRadius(radius=0.1): finds every point in the radius (can be slow on high-density data)
# best for: variable density scans where local scale matters
downpcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
o3d.visualization.draw_geometries([downpcd], point_show_normal=True)

Recompute the normal of the downsampled point cloud


In [1]:
with o3d.utility.VerbosityContextManager(
    o3d.utility.VerbosityLevel.Debug) as cm:
    labels = np.array(
        downpcd.cluster_dbscan(eps=0.02, min_points=10, print_progress=True))
max_label = labels.max()
print(f"point cloud has {max_label + 1} clusters")
colors = plt.get_cmap("tab20")(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0
downpcd.colors = o3d.utility.Vector3dVector(colors[:, :3])
o3d.visualization.draw_geometries([downpcd])

NameError: name 'o3d' is not defined